# 🧑‍💻 Multi-Agent Coding Solver
### 7 Agents · 7 Agentic AI Architectures · Template-Matched & Actually-Tested Code

This notebook implements a complete, self-contained multi-agent system for solving
common **coding problems** (two sum, reversing a string, palindrome check, Fibonacci,
factorial, prime check, FizzBuzz, binary search, sorting, anagram check, GCD, balanced
parentheses) — built to run directly in **Google Colab**.

**Key properties**

- ✅ **No paid LLM APIs** — every agent reasons using deterministic Python: keyword
  matching against a curated, hand-verified template library, and then **real code
  execution** against real test cases. "Verified" always means the code actually ran
  and passed, not that a model claimed it would.
- ✅ **7 specialized agents**, each with a single responsibility.
- ✅ **7 agentic architectures** — the same 7 agents are wired together in 7 different
  topologies (Single-Agent, Sequential, Hierarchical, Parallel, Collaborative, Debate,
  Supervisor), so you can directly compare how orchestration style affects the workflow.
- ✅ **Gradio UI** with a generated-code viewer, a per-test-case results table, a full
  agent execution trace, and independent verification.

---

## Table of Contents
1. Setup (Colab-ready)
2. The 7 Agents
3. The Coding Problem Template Library
4. The Matcher & Real Code Executor/Tester
5. Shared helpers
6. The 7 Agentic Architectures
7. Example questions
8. Gradio interactive app


## 1. Setup

Works out of the box in **Google Colab** — just run the cells top to bottom. No API
keys, no accounts, no paid services.


In [ ]:
# Install dependencies (safe to re-run; skips already-satisfied packages).
# No paid LLM API keys are required anywhere in this notebook.
%pip install -q gradio pandas


In [ ]:
import re
import time
import textwrap
import traceback
import concurrent.futures
from typing import List, Tuple, Dict, Any, Optional

import pandas as pd

print("Environment ready. Core libraries loaded (no paid LLM API required).")


Environment ready. Core libraries loaded (no paid LLM API required).


## 2. The 7 Agents

Each agent below owns exactly one responsibility:

| Agent | Responsibility |
|---|---|
| **CoordinationAgent** | Owns the overall workflow; decides call order; assembles results |
| **PlanningAgent** | Breaks the coding task into an ordered plan of steps |
| **ReasoningAgent** | Matches the request against known problem types via keyword scoring |
| **LogicAgent** | Retrieves the matching implementation and **actually executes it** against test cases |
| **PatternAgent** | Summarizes code size, pass/fail counts, and which tests (if any) are failing |
| **CriticAgent** | Looks for ambiguous matches or failing tests before declaring success |
| **VerifierAgent** | Independently re-runs the full test suite one more time for a binding verdict |

First, a small `ExecutionTrace` utility and a base `Agent` class that every agent
inherits from — this is what powers the "Agent Execution Trace" tab in the UI.


In [ ]:
class ExecutionTrace:
    """Collects a step-by-step log of what every agent does, so the
    reasoning process is fully transparent to the end user."""

    def __init__(self):
        self.steps: List[Dict[str, Any]] = []

    def log(self, architecture: str, agent: str, action: str, detail: str):
        self.steps.append({
            "step": len(self.steps) + 1,
            "architecture": architecture,
            "agent": agent,
            "action": action,
            "detail": detail,
            "timestamp": round(time.time(), 4),
        })

    def as_dataframe(self) -> pd.DataFrame:
        if not self.steps:
            return pd.DataFrame(columns=["step", "architecture", "agent", "action", "detail"])
        df = pd.DataFrame(self.steps)
        return df[["step", "architecture", "agent", "action", "detail"]]

    def reset(self):
        self.steps = []


class Agent:
    """Base class for every agent in the system. Each concrete agent
    overrides `run` and performs one well-defined piece of cognitive work.
    No paid LLM API calls are used anywhere -- every agent reasons using
    deterministic Python logic (regular expressions, constraint checking,
    permutation search, and rule-based heuristics)."""

    name: str = "Agent"

    def __init__(self, trace: Optional[ExecutionTrace] = None):
        self.trace = trace

    def log(self, architecture: str, action: str, detail: str):
        if self.trace is not None:
            self.trace.log(architecture, self.name, action, detail)

    def run(self, *args, **kwargs):
        raise NotImplementedError


In [ ]:
# ---------------------------------------------------------------------------
# 7 AGENTS
# ---------------------------------------------------------------------------
# Each agent owns a single responsibility. They are combined in different
# orders/topologies later to realize the 7 agentic architectures. No paid
# LLM API is used anywhere: code is selected from a curated, hand-verified
# template library and then REALLY EXECUTED against real test cases, so
# "verified" always means "actually ran and passed", not "an LLM claimed so".

class CoordinationAgent(Agent):
    """Owns the overall workflow: decides which agents to call, in what
    order, and assembles the final package of results."""
    name = "CoordinationAgent"

    def run(self, architecture: str, question: str, steps_planned: List[str]):
        self.log(architecture, "coordinate",
                  f"Received coding request ({len(question)} chars). Will coordinate "
                  f"{len(steps_planned)} planned steps: {', '.join(steps_planned)}.")
        return {"question": question, "plan": steps_planned}


class PlanningAgent(Agent):
    """Breaks the coding task down into an ordered plan of steps."""
    name = "PlanningAgent"

    DEFAULT_PLAN = [
        "understand_requirement",
        "select_algorithm",
        "generate_code",
        "run_tests",
        "analyze_results",
        "critique_code",
        "verify_final_solution",
    ]

    def run(self, architecture: str, question: str) -> List[str]:
        plan = list(self.DEFAULT_PLAN)
        self.log(architecture, "plan", f"Decomposed task into steps: {plan}")
        return plan


class ReasoningAgent(Agent):
    """Performs the initial semantic reasoning over the raw request: which
    known problem type does this most resemble, based on keyword matching
    against the template library."""
    name = "ReasoningAgent"

    def run(self, architecture: str, question: str) -> Dict[str, Any]:
        matches = match_templates(question)
        if matches:
            best, score = matches[0]
            self.log(architecture, "reason",
                      f"Matched request to problem type '{best['id']}' "
                      f"(keyword score={score}). {len(matches)-1} other candidate(s) "
                      f"also matched with lower confidence.")
        else:
            self.log(architecture, "reason",
                      "No known problem type matched this request against the template library.")
        return {"matches": matches}


class LogicAgent(Agent):
    """Formalizes the choice into actual code: retrieves the matched
    template's implementation and REALLY EXECUTES it against every test
    case (not a simulated or claimed result)."""
    name = "LogicAgent"

    def run(self, architecture: str, matches):
        if not matches:
            self.log(architecture, "generate_code", "No template available -- cannot generate code.")
            return {"template": None, "results": [], "all_passed": False, "error": "no matching template"}

        template, score = matches[0]
        self.log(architecture, "generate_code",
                  f"Generated {len(template['code'].splitlines())}-line implementation of "
                  f"`{template['function_name']}` from the '{template['id']}' template.")

        results, all_passed, err = run_tests(template["code"], template["function_name"], template["tests"])
        self.log(architecture, "run_tests",
                  f"Executed {len(results)} test case(s) against the generated code. "
                  f"all_passed={all_passed}." + (f" error={err}" if err else ""))
        return {"template": template, "results": results, "all_passed": all_passed, "error": err}


class PatternAgent(Agent):
    """Looks across the test results and the generated code to surface
    useful metadata: how many tests passed, code size, and which specific
    test cases (if any) are failing."""
    name = "PatternAgent"

    def run(self, architecture: str, template, results):
        if not template:
            self.log(architecture, "pattern_scan", "No code to analyze.")
            return {"note": "no code generated"}

        n_pass = sum(1 for r in results if r["passed"])
        n_total = len(results)
        loc = len(template["code"].splitlines())
        failing = [r["test"] for r in results if not r["passed"]]
        note = (f"{n_pass}/{n_total} test(s) passed. Code is {loc} line(s) long. "
                + (f"Failing: {failing}." if failing else "No failing tests."))
        self.log(architecture, "pattern_scan", note)
        return {"note": note, "n_pass": n_pass, "n_total": n_total, "loc": loc}


class CriticAgent(Agent):
    """Actively tries to poke holes in the generated solution: re-examines
    every test result individually, and flags low-confidence matches or
    partial test failures rather than declaring success prematurely."""
    name = "CriticAgent"

    def run(self, architecture: str, matches, template, results, all_passed):
        issues = []
        if template is None:
            issues.append("No template matched the request -- nothing to critique or ship.")
        else:
            top_score = matches[0][1]
            if top_score == 1 and len(matches) > 1 and matches[1][1] == top_score:
                issues.append("Multiple problem types matched with equal confidence -- "
                               "the request may be ambiguous.")
            if all_passed:
                issues.append(f"All {len(results)} test case(s) passed -- no correctness issues found.")
            else:
                failing = [r for r in results if not r["passed"]]
                issues.append(f"{len(failing)} test case(s) FAILED -- solution is not yet correct.")
        verdict = "; ".join(issues)
        self.log(architecture, "critique", verdict)
        return {"issues": issues}


class VerifierAgent(Agent):
    """Performs the final, independent verification: re-runs the full test
    suite one more time against the generated code and reports a clear,
    binding pass/fail verdict."""
    name = "VerifierAgent"

    def run(self, architecture: str, template, tests_already_run):
        if template is None:
            self.log(architecture, "verify", "Nothing to verify -- no code was generated.")
            return {"verified": False, "detail": "No matching template / no code generated."}

        results, all_passed, err = run_tests(template["code"], template["function_name"], template["tests"])
        detail = (f"Independent re-run of all {len(results)} test case(s): "
                   f"{'ALL PASSED' if all_passed else 'AT LEAST ONE FAILED'}.")
        self.log(architecture, "verify", detail)
        return {"verified": all_passed, "detail": detail, "results": results}


## 3. The Coding Problem Template Library

Each template is a small, **hand-written and hand-verified** algorithm implementation
plus its own test cases. This is the "ground truth" the whole system draws from — the
point of using a template library instead of an LLM is that every answer can be
**genuinely executed and checked**, not just asserted.

12 problem types are included: Two Sum, Reverse String, Palindrome Check, Fibonacci,
Factorial, Prime Check, FizzBuzz, Binary Search, Sort a List, Anagram Check, GCD, and
Balanced Parentheses.


In [ ]:
# ---------------------------------------------------------------------------
# TEMPLATE LIBRARY
# Each template is a known, correct, hand-written algorithm implementation
# (this is what lets the whole pipeline verify results for real -- by
# actually EXECUTING the code against test cases -- instead of trusting an
# LLM's unverified guess).
# ---------------------------------------------------------------------------

TEMPLATES = [
    {
        "id": "two_sum",
        "keywords": ["two sum", "pair that adds", "pair sums to", "indices of two numbers"],
        "function_name": "two_sum",
        "description": "Given a list of numbers and a target, return indices of the two numbers that add up to the target.",
        "code": textwrap.dedent("""
            def two_sum(nums, target):
                seen = {}
                for i, n in enumerate(nums):
                    complement = target - n
                    if complement in seen:
                        return [seen[complement], i]
                    seen[n] = i
                return []
        """).strip(),
        "tests": [
            {"args": ([2, 7, 11, 15], 9), "expected": [0, 1]},
            {"args": ([3, 2, 4], 6), "expected": [1, 2]},
            {"args": ([1, 5, 3, 8], 11), "expected": [2, 3]},
        ],
    },
    {
        "id": "reverse_string",
        "keywords": ["reverse a string", "reverse string", "reverse the string", "reverse a given string"],
        "function_name": "reverse_string",
        "description": "Reverse a string.",
        "code": textwrap.dedent("""
            def reverse_string(s):
                return s[::-1]
        """).strip(),
        "tests": [
            {"args": ("hello",), "expected": "olleh"},
            {"args": ("Python",), "expected": "nohtyP"},
            {"args": ("",), "expected": ""},
        ],
    },
    {
        "id": "palindrome_check",
        "keywords": ["palindrome"],
        "function_name": "is_palindrome",
        "description": "Check whether a string reads the same forwards and backwards.",
        "code": textwrap.dedent("""
            def is_palindrome(s):
                cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())
                return cleaned == cleaned[::-1]
        """).strip(),
        "tests": [
            {"args": ("racecar",), "expected": True},
            {"args": ("A man a plan a canal Panama",), "expected": True},
            {"args": ("hello",), "expected": False},
        ],
    },
    {
        "id": "fibonacci",
        "keywords": ["fibonacci"],
        "function_name": "fibonacci",
        "description": "Return the nth Fibonacci number (0-indexed).",
        "code": textwrap.dedent("""
            def fibonacci(n):
                if n < 0:
                    raise ValueError("n must be non-negative")
                a, b = 0, 1
                for _ in range(n):
                    a, b = b, a + b
                return a
        """).strip(),
        "tests": [
            {"args": (0,), "expected": 0},
            {"args": (1,), "expected": 1},
            {"args": (10,), "expected": 55},
        ],
    },
    {
        "id": "factorial",
        "keywords": ["factorial"],
        "function_name": "factorial",
        "description": "Compute the factorial of a non-negative integer.",
        "code": textwrap.dedent("""
            def factorial(n):
                if n < 0:
                    raise ValueError("n must be non-negative")
                result = 1
                for i in range(2, n + 1):
                    result *= i
                return result
        """).strip(),
        "tests": [
            {"args": (0,), "expected": 1},
            {"args": (5,), "expected": 120},
            {"args": (7,), "expected": 5040},
        ],
    },
    {
        "id": "prime_check",
        "keywords": ["prime number", "is prime", "check if prime", "check prime"],
        "function_name": "is_prime",
        "description": "Check whether a number is prime.",
        "code": textwrap.dedent("""
            def is_prime(n):
                if n < 2:
                    return False
                if n in (2, 3):
                    return True
                if n % 2 == 0:
                    return False
                i = 3
                while i * i <= n:
                    if n % i == 0:
                        return False
                    i += 2
                return True
        """).strip(),
        "tests": [
            {"args": (2,), "expected": True},
            {"args": (17,), "expected": True},
            {"args": (18,), "expected": False},
            {"args": (1,), "expected": False},
        ],
    },
    {
        "id": "fizzbuzz",
        "keywords": ["fizzbuzz", "fizz buzz"],
        "function_name": "fizzbuzz",
        "description": "Return the FizzBuzz sequence from 1 to n.",
        "code": textwrap.dedent("""
            def fizzbuzz(n):
                out = []
                for i in range(1, n + 1):
                    if i % 15 == 0:
                        out.append("FizzBuzz")
                    elif i % 3 == 0:
                        out.append("Fizz")
                    elif i % 5 == 0:
                        out.append("Buzz")
                    else:
                        out.append(str(i))
                return out
        """).strip(),
        "tests": [
            {"args": (15,), "expected": ["1","2","Fizz","4","Buzz","Fizz","7","8","Fizz","Buzz","11","Fizz","13","14","FizzBuzz"]},
            {"args": (5,), "expected": ["1","2","Fizz","4","Buzz"]},
        ],
    },
    {
        "id": "binary_search",
        "keywords": ["binary search"],
        "function_name": "binary_search",
        "description": "Search a sorted list for a target and return its index, or -1 if absent.",
        "code": textwrap.dedent("""
            def binary_search(sorted_list, target):
                lo, hi = 0, len(sorted_list) - 1
                while lo <= hi:
                    mid = (lo + hi) // 2
                    if sorted_list[mid] == target:
                        return mid
                    elif sorted_list[mid] < target:
                        lo = mid + 1
                    else:
                        hi = mid - 1
                return -1
        """).strip(),
        "tests": [
            {"args": ([1, 3, 5, 7, 9, 11], 7), "expected": 3},
            {"args": ([1, 3, 5, 7, 9, 11], 2), "expected": -1},
            {"args": ([2, 4, 6, 8], 2), "expected": 0},
        ],
    },
    {
        "id": "sort_list",
        "keywords": ["sort a list", "sort the list", "sort an array", "bubble sort"],
        "function_name": "sort_list",
        "description": "Sort a list of numbers in ascending order.",
        "code": textwrap.dedent("""
            def sort_list(nums):
                arr = list(nums)
                n = len(arr)
                for i in range(n):
                    for j in range(0, n - i - 1):
                        if arr[j] > arr[j + 1]:
                            arr[j], arr[j + 1] = arr[j + 1], arr[j]
                return arr
        """).strip(),
        "tests": [
            {"args": ([5, 2, 9, 1, 5, 6],), "expected": [1, 2, 5, 5, 6, 9]},
            {"args": ([3, 1, 2],), "expected": [1, 2, 3]},
            {"args": ([],), "expected": []},
        ],
    },
    {
        "id": "anagram_check",
        "keywords": ["anagram"],
        "function_name": "is_anagram",
        "description": "Check whether two strings are anagrams of each other.",
        "code": textwrap.dedent("""
            def is_anagram(a, b):
                a_clean = sorted(a.replace(" ", "").lower())
                b_clean = sorted(b.replace(" ", "").lower())
                return a_clean == b_clean
        """).strip(),
        "tests": [
            {"args": ("listen", "silent"), "expected": True},
            {"args": ("hello", "world"), "expected": False},
            {"args": ("Dormitory", "Dirty Room"), "expected": True},
        ],
    },
    {
        "id": "gcd",
        "keywords": ["gcd", "greatest common divisor", "hcf"],
        "function_name": "gcd",
        "description": "Compute the greatest common divisor of two integers.",
        "code": textwrap.dedent("""
            def gcd(a, b):
                while b:
                    a, b = b, a % b
                return abs(a)
        """).strip(),
        "tests": [
            {"args": (48, 18), "expected": 6},
            {"args": (17, 5), "expected": 1},
            {"args": (100, 75), "expected": 25},
        ],
    },
    {
        "id": "balanced_parentheses",
        "keywords": ["balanced parentheses", "balanced brackets", "valid parentheses",
                     "parentheses and brackets", "parentheses is balanced", "brackets is balanced"],
        "function_name": "is_balanced",
        "description": "Check whether a string of brackets is balanced.",
        "code": textwrap.dedent("""
            def is_balanced(s):
                pairs = {')': '(', ']': '[', '}': '{'}
                stack = []
                for ch in s:
                    if ch in '([{':
                        stack.append(ch)
                    elif ch in ')]}':
                        if not stack or stack.pop() != pairs[ch]:
                            return False
                return not stack
        """).strip(),
        "tests": [
            {"args": ("()[]{}",), "expected": True},
            {"args": ("(]",), "expected": False},
            {"args": ("([{}])",), "expected": True},
        ],
    },
]

TEMPLATE_BY_ID = {t["id"]: t for t in TEMPLATES}


## 4. The Matcher & Real Code Executor/Tester

`match_templates()` scores every template by how many of its keyword phrases appear in
the user's question and ranks them best-first — this is a lightweight stand-in for
intent classification, done without any external API call.

`run_tests()` does the important part: it `exec()`s the generated code in a fresh
namespace and calls the resulting function with each test case's real arguments,
comparing the real output to the expected output. Nothing here is simulated.


In [ ]:


def match_templates(question: str):
    """Score every template by how many of its keyword phrases appear in the
    question, and return them ranked best-first. Returns a list of
    (template, score) tuples with score > 0."""
    q = question.lower()
    scored = []
    for t in TEMPLATES:
        score = sum(1 for kw in t["keywords"] if kw in q)
        if score > 0:
            scored.append((t, score))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored


def run_tests(code: str, function_name: str, tests):
    """Actually EXECUTE the generated code in a restricted namespace and run
    every test case against it. Returns a list of per-test result dicts and
    an overall pass/fail."""
    namespace = {}
    try:
        exec(code, namespace)
    except Exception as e:
        return [], False, f"Code failed to compile/execute: {e}"

    fn = namespace.get(function_name)
    if fn is None:
        return [], False, f"Function '{function_name}' was not defined by the generated code."

    results = []
    all_passed = True
    for i, test in enumerate(tests, start=1):
        args = test["args"]
        expected = test["expected"]
        try:
            actual = fn(*args)
            passed = actual == expected
        except Exception as e:
            actual = f"EXCEPTION: {e}"
            passed = False
        if not passed:
            all_passed = False
        results.append({
            "test": f"Test {i}",
            "args": repr(args),
            "expected": repr(expected),
            "actual": repr(actual),
            "passed": passed,
        })
    return results, all_passed, None


## 5. Shared helpers

Small utilities used by every architecture to render results consistently.

In [ ]:
def results_to_dataframe(results: List[Dict[str, Any]]) -> pd.DataFrame:
    """Turns a list of per-test-case result dicts into a readable table."""
    if not results:
        return pd.DataFrame(columns=["test", "args", "expected", "actual", "passed"])
    return pd.DataFrame(results)[["test", "args", "expected", "actual", "passed"]]


def format_final_answer(template, all_passed: bool, results) -> str:
    if template is None:
        return "No known problem type matched this request -- no code was generated."
    n_pass = sum(1 for r in results if r["passed"])
    status = "ALL TESTS PASSED" if all_passed else f"{n_pass}/{len(results)} TESTS PASSED"
    return (f"Problem type: {template['id']}  |  Function: `{template['function_name']}`\n"
            f"{template['description']}\n"
            f"Status: {status}")


def format_code(template) -> str:
    if template is None:
        return "# No code generated -- no template matched this request."
    return template["code"]


## 6. The 7 Agentic Architectures

The same 7 agents are now composed in 7 different ways. Each `run_*` function takes
the raw coding question and returns a consistent result dictionary (trace, generated
code, test results, final answer, verification status) so the UI can treat them
interchangeably — but *how* they get there differs meaningfully.

### 6.1 Single-Agent
One generalist agent does every step itself, with no delegation at all. This is the
simplest possible design and serves as a baseline.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 1: SINGLE-AGENT
# ---------------------------------------------------------------------------
# One generalist agent does every step itself, start to finish. No
# delegation, no hand-offs. Simplest possible architecture -- useful as a
# baseline to compare against the multi-agent designs below.

class SoloAgent(Agent):
    name = "SoloAgent"

    def run(self, architecture: str, question: str):
        matches = match_templates(question)
        if matches:
            best, score = matches[0]
            self.log(architecture, "reason", f"Matched problem type '{best['id']}' (score={score}).")
        else:
            self.log(architecture, "reason", "No known problem type matched.")

        template = matches[0][0] if matches else None
        if template:
            results, all_passed, err = run_tests(template["code"], template["function_name"], template["tests"])
            self.log(architecture, "generate_and_test",
                      f"Generated code for '{template['id']}' and ran {len(results)} test(s); "
                      f"all_passed={all_passed}.")
        else:
            results, all_passed, err = [], False, "no matching template"
            self.log(architecture, "generate_and_test", "Skipped -- no template to generate from.")

        n_pass = sum(1 for r in results if r["passed"])
        self.log(architecture, "pattern_scan", f"{n_pass}/{len(results)} test(s) passed.")
        self.log(architecture, "critique", "ALL PASSED" if all_passed else "Some test(s) failed or no template matched.")
        self.log(architecture, "verify", "PASSED" if all_passed else "NOT VERIFIED")

        return matches, template, results, all_passed


def run_single_agent(question: str) -> Dict[str, Any]:
    arch = "Single-Agent"
    trace = ExecutionTrace()
    agent = SoloAgent(trace)
    matches, template, results, all_passed = agent.run(arch, question)
    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(template),
        "results": results_to_dataframe(results),
        "final_answer": format_final_answer(template, all_passed, results),
        "verified": all_passed,
        "extra_notes": "A single generalist agent performed matching, code generation, "
                        "testing, and verification entirely on its own.",
    }


### 6.2 Sequential
A strict pipeline: `Coordination → Planning → Reasoning → Logic → Pattern → Critic →
Verifier`. Each agent's output feeds directly into the next, like an assembly line.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 2: SEQUENTIAL
# ---------------------------------------------------------------------------
# A strict pipeline: Coordination -> Planning -> Reasoning -> Logic ->
# Pattern -> Critic -> Verifier. Each agent's output becomes the next
# agent's input, one after another, like an assembly line.

def run_sequential(question: str) -> Dict[str, Any]:
    arch = "Sequential"
    trace = ExecutionTrace()

    coordinator = CoordinationAgent(trace)
    planner = PlanningAgent(trace)
    reasoner = ReasoningAgent(trace)
    logician = LogicAgent(trace)
    patterner = PatternAgent(trace)
    critic = CriticAgent(trace)
    verifier = VerifierAgent(trace)

    plan = planner.run(arch, question)
    coordinator.run(arch, question, plan)

    reason_out = reasoner.run(arch, question)
    matches = reason_out["matches"]

    logic_out = logician.run(arch, matches)
    template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]

    pattern_out = patterner.run(arch, template, results)
    critic_out = critic.run(arch, matches, template, results, all_passed)
    verify_out = verifier.run(arch, template, results)

    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(template),
        "results": results_to_dataframe(results),
        "final_answer": format_final_answer(template, all_passed, results),
        "verified": verify_out["verified"],
        "extra_notes": f"Pattern: {pattern_out['note']} | Critic: {'; '.join(critic_out['issues'])}",
    }


### 6.3 Hierarchical
A root `CoordinationAgent` delegates to branch managers — an *Understanding* branch
(`ReasoningAgent`), a *Building* branch (`LogicAgent` + `PatternAgent`), and a *Review*
branch (`CriticAgent` + `VerifierAgent`) — and makes the final call once all branches
report back. This models a delegated org-chart, not a flat pipeline.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 3: HIERARCHICAL
# ---------------------------------------------------------------------------
# A root Coordination agent delegates to branch managers: an Understanding
# branch (Reasoning), a Building branch (Logic + Pattern), and a Review
# branch (Critic + Verifier). The root only sees each branch's summarized
# report, not every low-level step -- an org-chart, not a flat pipeline.

def run_hierarchical(question: str) -> Dict[str, Any]:
    arch = "Hierarchical"
    trace = ExecutionTrace()

    root = CoordinationAgent(trace)
    reasoner = ReasoningAgent(trace)
    logician = LogicAgent(trace)
    patterner = PatternAgent(trace)
    critic = CriticAgent(trace)
    verifier = VerifierAgent(trace)

    root.run(arch, question, ["understanding_branch", "building_branch", "review_branch"])
    root.log(arch, "delegate", "Delegating problem understanding to ReasoningAgent (branch manager).")

    reason_out = reasoner.run(arch, question)
    matches = reason_out["matches"]
    root.log(arch, "receive",
              f"Understanding branch reported back: "
              f"{matches[0][0]['id'] if matches else 'no match'}.")

    root.log(arch, "delegate", "Delegating code generation + testing to LogicAgent (branch manager).")
    logic_out = logician.run(arch, matches)
    template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]
    pattern_out = patterner.run(arch, template, results)
    root.log(arch, "receive", f"Building branch reported all_passed={all_passed}. {pattern_out['note']}")

    root.log(arch, "delegate", "Delegating independent review to CriticAgent and VerifierAgent.")
    critic_out = critic.run(arch, matches, template, results, all_passed)
    verify_out = verifier.run(arch, template, results)
    root.log(arch, "receive", f"Review branch verdict: verified={verify_out['verified']}.")

    root.log(arch, "finalize", "Root coordinator approved the final answer from the review branch.")

    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(template),
        "results": results_to_dataframe(results),
        "final_answer": format_final_answer(template, all_passed, results),
        "verified": verify_out["verified"],
        "extra_notes": f"Pattern: {pattern_out['note']} | Critic: {'; '.join(critic_out['issues'])}",
    }


### 6.4 Parallel
Independent sub-tasks genuinely run **concurrently** using a `ThreadPoolExecutor`:
Planning + Reasoning run together first, and later Pattern-analysis + Critic-review
run together, since neither pair depends on the other within its stage.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 4: PARALLEL
# ---------------------------------------------------------------------------
# Independent sub-tasks run concurrently instead of one-at-a-time.
# Stage A: Planning and Reasoning run in parallel (neither depends on the
#          other's output).
# Stage B: Logic generates + executes the code (needs Stage A's output).
# Stage C: Pattern analysis and Critic review run in parallel (both only
#          need the test results, not each other).
# Stage D: Verifier finalizes.
# A real ThreadPoolExecutor is used so the branches genuinely execute
# concurrently, not just conceptually.

def run_parallel(question: str) -> Dict[str, Any]:
    arch = "Parallel"
    trace = ExecutionTrace()

    planner = PlanningAgent(trace)
    reasoner = ReasoningAgent(trace)
    logician = LogicAgent(trace)
    patterner = PatternAgent(trace)
    critic = CriticAgent(trace)
    verifier = VerifierAgent(trace)

    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
        fut_plan = pool.submit(planner.run, arch, question)
        fut_reason = pool.submit(reasoner.run, arch, question)
        plan = fut_plan.result()
        reason_out = fut_reason.result()

    matches = reason_out["matches"]
    logic_out = logician.run(arch, matches)
    template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]

    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
        fut_pattern = pool.submit(patterner.run, arch, template, results)
        fut_critic = pool.submit(critic.run, arch, matches, template, results, all_passed)
        pattern_out = fut_pattern.result()
        critic_out = fut_critic.result()

    verify_out = verifier.run(arch, template, results)

    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(template),
        "results": results_to_dataframe(results),
        "final_answer": format_final_answer(template, all_passed, results),
        "verified": verify_out["verified"],
        "extra_notes": f"Ran Planning+Reasoning concurrently, then Pattern+Critic concurrently. "
                        f"Pattern: {pattern_out['note']} | Critic: {'; '.join(critic_out['issues'])}",
    }


### 6.5 Collaborative
All agents read from and write to one shared **blackboard** dictionary. Later agents
(e.g. `CriticAgent`) can pick up and react to findings that earlier agents posted (e.g.
`PatternAgent`'s pass/fail counts), rather than following a rigid hand-off order.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 5: COLLABORATIVE (shared blackboard)
# ---------------------------------------------------------------------------
# All agents read from and write to one shared "blackboard" dictionary.
# Rather than a rigid hand-off order, each agent contributes whatever it can
# whenever its inputs are available -- including later agents (Pattern,
# Critic) posting findings that earlier-style agents can react to.

def run_collaborative(question: str) -> Dict[str, Any]:
    arch = "Collaborative"
    trace = ExecutionTrace()

    reasoner = ReasoningAgent(trace)
    logician = LogicAgent(trace)
    patterner = PatternAgent(trace)
    critic = CriticAgent(trace)
    verifier = VerifierAgent(trace)

    blackboard: Dict[str, Any] = {"question": question}

    reason_out = reasoner.run(arch, question)
    blackboard.update(reason_out)
    blackboard["note_reasoning"] = "ReasoningAgent posted candidate problem-type matches to the blackboard."

    logic_out = logician.run(arch, blackboard["matches"])
    blackboard.update(logic_out)

    pattern_out = patterner.run(arch, blackboard["template"], blackboard["results"])
    blackboard["pattern"] = pattern_out
    if blackboard.get("template") and pattern_out.get("n_pass", 0) < pattern_out.get("n_total", 0):
        CriticAgent(trace).log(
            arch, "read_blackboard",
            f"CriticAgent reads PatternAgent's finding that only "
            f"{pattern_out['n_pass']}/{pattern_out['n_total']} tests passed and "
            f"escalates this as the top priority issue.")

    critic_out = critic.run(arch, blackboard["matches"], blackboard["template"],
                             blackboard["results"], blackboard["all_passed"])
    blackboard["critic"] = critic_out

    verify_out = verifier.run(arch, blackboard["template"], blackboard["results"])
    blackboard["verification"] = verify_out

    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(blackboard["template"]),
        "results": results_to_dataframe(blackboard["results"]),
        "final_answer": format_final_answer(blackboard["template"], blackboard["all_passed"], blackboard["results"]),
        "verified": verify_out["verified"],
        "extra_notes": f"Shared blackboard had {len(blackboard)} entries by the end. "
                        f"Pattern: {pattern_out['note']} | Critic: {'; '.join(critic_out['issues'])}",
    }


### 6.6 Debate
An **Advocate** (`Reasoning` + `Logic`) proposes the top-ranked problem-type match; a
**Skeptic** (`CriticAgent`) challenges it. If the top match fails its tests, the
Advocate gets a second round to fall back to the next-best match. A neutral
`VerifierAgent` then acts as judge and delivers the binding final verdict.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 6: DEBATE
# ---------------------------------------------------------------------------
# Two "sides" argue: an Advocate (Reasoning + Logic) proposes a solution,
# and a Skeptic (Critic) tries to tear it apart. If the Skeptic finds
# failing tests, the Advocate gets a second round to try the next-best
# matching template. A neutral VerifierAgent then acts as judge and
# delivers the final, binding verdict.

def run_debate(question: str) -> Dict[str, Any]:
    arch = "Debate"
    trace = ExecutionTrace()

    reasoner = ReasoningAgent(trace)
    logician = LogicAgent(trace)
    critic = CriticAgent(trace)
    verifier = VerifierAgent(trace)
    patterner = PatternAgent(trace)

    reason_out = reasoner.run(arch, question)
    matches = reason_out["matches"]

    logician.log(arch, "advocate", "Advocate (Reasoning+Logic) proposes the top-ranked problem-type match.")
    logic_out = logician.run(arch, matches)
    template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]

    round_num = 1
    critic_out = critic.run(arch, matches, template, results, all_passed)
    critic.log(arch, "challenge", f"Round {round_num}: Skeptic challenges the proposal -- "
                                    f"{'; '.join(critic_out['issues'])}")

    if not all_passed and len(matches) > 1:
        round_num = 2
        logician.log(arch, "defend",
                      f"Round {round_num}: Advocate concedes the top match failed testing and "
                      f"falls back to the next-ranked candidate '{matches[1][0]['id']}'.")
        logic_out = logician.run(arch, [matches[1]])
        template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]
        critic_out = critic.run(arch, [matches[1]], template, results, all_passed)

    pattern_out = patterner.run(arch, template, results)
    verify_out = verifier.run(arch, template, results)
    verifier.log(arch, "judge", f"Acting as neutral judge after {round_num} round(s) of debate: "
                                  f"{verify_out['detail']}")

    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(template),
        "results": results_to_dataframe(results),
        "final_answer": format_final_answer(template, all_passed, results),
        "verified": verify_out["verified"],
        "extra_notes": f"Debate ran {round_num} round(s). Pattern: {pattern_out['note']} "
                        f"| Skeptic: {'; '.join(critic_out['issues'])}",
    }


### 6.7 Supervisor
A `SupervisorAgent` actively monitors the worker agents and can **intervene** — for
example, retrying with the next-ranked template match if the first one fails testing,
or flagging when no template matches at all — rather than passively watching a
pipeline run to completion.


In [ ]:
# ---------------------------------------------------------------------------
# ARCHITECTURE 7: SUPERVISOR
# ---------------------------------------------------------------------------
# A SupervisorAgent actively monitors worker agents and can order a retry
# if quality checks fail -- for example, falling back to the next-ranked
# template match if the first one fails its tests, or flagging when no
# template matches at all. Unlike Hierarchical (fixed delegation tree) or
# Debate (two opposing sides), the Supervisor's job is ongoing quality
# control with the power to intervene mid-workflow.

class SupervisorAgent(CoordinationAgent):
    name = "SupervisorAgent"


def run_supervisor(question: str) -> Dict[str, Any]:
    arch = "Supervisor"
    trace = ExecutionTrace()

    supervisor = SupervisorAgent(trace)
    reasoner = ReasoningAgent(trace)
    logician = LogicAgent(trace)
    patterner = PatternAgent(trace)
    critic = CriticAgent(trace)
    verifier = VerifierAgent(trace)

    supervisor.log(arch, "monitor_start", "Supervisor begins monitoring the worker agents.")

    reason_out = reasoner.run(arch, question)
    matches = reason_out["matches"]
    if not matches:
        supervisor.log(arch, "intervene",
                        "No problem-type match found -- supervisor flags this run as unresolved "
                        "and will not attempt code generation.")

    logic_out = logician.run(arch, matches)
    template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]

    attempt = 1
    tried = {matches[0][0]["id"]} if matches else set()
    while not all_passed and template is not None and len(matches) > attempt:
        supervisor.log(arch, "intervene",
                        f"Attempt {attempt} failed testing -- supervisor orders a retry with the "
                        f"next-ranked candidate template.")
        attempt += 1
        candidate = matches[attempt - 1]
        logic_out = logician.run(arch, [candidate])
        template, results, all_passed = logic_out["template"], logic_out["results"], logic_out["all_passed"]
        tried.add(candidate[0]["id"])

    if attempt > 1:
        supervisor.log(arch, "retry_result",
                        f"After {attempt} attempt(s) (tried {sorted(tried)}), "
                        f"all_passed={all_passed}.")

    pattern_out = patterner.run(arch, template, results)
    critic_out = critic.run(arch, matches, template, results, all_passed)
    verify_out = verifier.run(arch, template, results)
    supervisor.log(arch, "monitor_end", f"Supervisor closes out the run. verified={verify_out['verified']}.")

    return {
        "architecture": arch,
        "trace": trace.as_dataframe(),
        "code": format_code(template),
        "results": results_to_dataframe(results),
        "final_answer": format_final_answer(template, all_passed, results),
        "verified": verify_out["verified"],
        "extra_notes": f"Pattern: {pattern_out['note']} | Critic: {'; '.join(critic_out['issues'])}",
    }


## 7. Example Questions

A registry of architectures plus ready-to-run example coding questions — including one intentionally unrelated question, to demonstrate graceful failure handling when nothing matches.

In [ ]:
ARCHITECTURES = {
    "Single-Agent": run_single_agent,
    "Sequential": run_sequential,
    "Hierarchical": run_hierarchical,
    "Parallel": run_parallel,
    "Collaborative": run_collaborative,
    "Debate": run_debate,
    "Supervisor": run_supervisor,
}

EXAMPLE_QUESTIONS = [
    "Write a function to check if a number is prime.",
    "Reverse a given string.",
    "Solve the classic two sum problem: given a list of numbers and a target, "
    "return the indices of the two numbers that add up to the target.",
    "Check if a given word or phrase is a palindrome.",
    "Compute the nth Fibonacci number.",
    "Sort a list of numbers in ascending order.",
    "Check whether a string of parentheses and brackets is balanced.",
    "Find the greatest common divisor (GCD) of two integers.",
    "Implement binary search on a sorted list.",
    "What's the best way to organize a birthday party?",  # intentionally unmatched, to demo graceful failure handling
]

print(f"Loaded {len(ARCHITECTURES)} architectures, {len(TEMPLATES)} problem templates, "
      f"and {len(EXAMPLE_QUESTIONS)} example questions.")


Loaded 7 architectures, 12 problem templates, and 10 example questions.


### Quick sanity check (optional)
Run every architecture against every example question and print a one-line summary.
This is a good way to confirm your environment is working before opening the UI.


In [ ]:
for arch_name, arch_fn in ARCHITECTURES.items():
    for q in EXAMPLE_QUESTIONS:
        result = arch_fn(q)
        n_tests = len(result['results'])
        print(f"{arch_name:14s} | verified={str(result['verified']):5s} | "
              f"{n_tests} test(s) | {q[:55]}...")


Single-Agent   | verified=True  | 4 test(s) | Write a function to check if a number is prime....
Single-Agent   | verified=True  | 3 test(s) | Reverse a given string....
Single-Agent   | verified=True  | 3 test(s) | Solve the classic two sum problem: given a list of numb...
Single-Agent   | verified=True  | 3 test(s) | Check if a given word or phrase is a palindrome....
Single-Agent   | verified=True  | 3 test(s) | Compute the nth Fibonacci number....
Single-Agent   | verified=True  | 3 test(s) | Sort a list of numbers in ascending order....
Single-Agent   | verified=True  | 3 test(s) | Check whether a string of parentheses and brackets is b...
Single-Agent   | verified=True  | 3 test(s) | Find the greatest common divisor (GCD) of two integers....
Single-Agent   | verified=True  | 3 test(s) | Implement binary search on a sorted list....
Single-Agent   | verified=False | 0 test(s) | What's the best way to organize a birthday party?...
Sequential     | verified=True  | 4 test(s) | Write 

## 8. Gradio Interactive App

Pick an architecture, type (or click an example) a coding problem, and click **Solve
with Agents**. You'll get:

- The **final answer**, with a ✅/⚠️ verification badge
- The **generated code** (a real, syntactically valid, executed Python function)
- A **test results** table showing every test case's args, expected output, actual
  output, and pass/fail
- A full **agent execution trace** — every action every agent took, in order

Running the cell below launches a local Gradio server. **In Google Colab**, change
`demo.launch()` to `demo.launch(share=True)` (edit the last line of the cell, or just
re-run with that change) to get a public shareable link, since Colab can't expose a
local port directly.


In [ ]:
import gradio as gr

CUSTOM_CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
#title-md h1 {margin-bottom: 0.1em;}
#answer-box textarea {font-size: 1.05em; font-weight: 600;}
footer {visibility: hidden}
"""


def solve_problem(question: str, architecture: str):
    if not question or not question.strip():
        empty_df = pd.DataFrame()
        return ("Please enter a coding problem, or pick an example below.",
                "# No code generated yet.", empty_df, empty_df, "")

    result = ARCHITECTURES[architecture](question)

    verdict_icon = "✅ VERIFIED" if result["verified"] else "⚠️ NOT VERIFIED"
    answer_text = f"{verdict_icon}\n\n{result['final_answer']}\n\n{result['extra_notes']}"

    return answer_text, result["code"], result["results"], result["trace"], architecture


def load_example(example_text):
    return example_text


with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="slate")) as demo:
    gr.Markdown(
        "# 🧑‍💻 Multi-Agent Coding Solver\n"
        "### 7 Agents · 7 Agentic Architectures · Template-Matched, Actually-Executed & Tested Code\n"
        "_No paid LLM APIs used — code is selected from a curated, hand-verified template "
        "library and then REALLY executed against real test cases, so \"verified\" always "
        "means it actually ran and passed._",
        elem_id="title-md",
    )

    with gr.Row():
        with gr.Column(scale=2):
            question_box = gr.Textbox(
                label="Coding Problem",
                placeholder="e.g. Write a function to check if a number is prime.",
                lines=5,
            )
            architecture_dd = gr.Dropdown(
                choices=list(ARCHITECTURES.keys()),
                value="Sequential",
                label="Agentic Architecture",
            )
            solve_btn = gr.Button("🧠 Solve with Agents", variant="primary")

            gr.Markdown("**Example questions** (click to load):")
            for ex in EXAMPLE_QUESTIONS:
                ex_btn = gr.Button(ex[:70] + ("..." if len(ex) > 70 else ""), size="sm")
                ex_btn.click(fn=load_example, inputs=gr.State(ex), outputs=question_box)

        with gr.Column(scale=3):
            answer_box = gr.Textbox(label="Final Answer & Verification", lines=5, elem_id="answer-box")
            with gr.Tab("Generated Code"):
                code_box = gr.Code(label="Generated solution", language="python")
            with gr.Tab("Test Results"):
                results_table = gr.Dataframe(label="Per-test-case execution results")
            with gr.Tab("Agent Execution Trace"):
                trace_table = gr.Dataframe(label="Step-by-step agent execution trace")
            arch_used = gr.Textbox(label="Architecture Used", interactive=False)

    solve_btn.click(
        fn=solve_problem,
        inputs=[question_box, architecture_dd],
        outputs=[answer_box, code_box, results_table, trace_table, arch_used],
    )

    gr.Markdown(
        "---\n"
        "**How it works:** `ReasoningAgent` matches the request against a library of known "
        "problem types, `LogicAgent` retrieves the matching implementation and **actually "
        "executes it** against real test cases, `PatternAgent` summarizes code size and "
        "pass/fail counts, `CriticAgent` looks for ambiguous matches or failing tests, and "
        "`VerifierAgent` independently re-runs the full test suite one more time. "
        "`CoordinationAgent` / `PlanningAgent` / `SupervisorAgent` orchestrate these workers "
        "differently depending on the architecture you select above — the Debate and "
        "Supervisor architectures will even fall back to the next-best-matching template if "
        "the first one fails its tests."
    )

# demo.launch(share=False)  # uncomment to launch when running this notebook locally
# In Google Colab, use demo.launch(share=True) to get a public shareable link.
demo.launch()


/tmp/ipykernel_1039/4095504288.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="slate")) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://48ab01e12308241a3e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
